# DSFB SRD Figure Regeneration

This notebook runs the `dsfb-srd-generate` crate from scratch inside Colab, then reads the newly created `output-dsfb-srd/<timestamp>/` directory and regenerates the three paper-facing figures for the deterministic Structural Regime Dynamics demonstrator.

The setup cell is intentionally crate-first: it clones the DSFB repository if needed, installs Rust if needed, executes the Rust binary, and plots the fresh empirical outputs from that run.



In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path("/content/dsfb")
AUTO_CLONE_REPO = True
DSFB_REPO_URL = "https://github.com/infinityabundance/dsfb.git"

def run_checked(command, cwd=None, env=None):
    print("+", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def unique_paths(paths):
    ordered = []
    seen = set()
    for candidate in paths:
        candidate = Path(candidate).expanduser()
        if candidate not in seen:
            seen.add(candidate)
            ordered.append(candidate)
    return ordered

def resolve_repo_root():
    candidates = []

    if REPO_ROOT is not None:
        candidates.append(Path(REPO_ROOT).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path("/content/dsfb"))

    for candidate in unique_paths(candidates):
        if (candidate / "crates/dsfb-srd/Cargo.toml").exists():
            return candidate

    return None

def ensure_repo_root():
    repo_root = resolve_repo_root()
    if repo_root is not None:
        return repo_root

    if not AUTO_CLONE_REPO:
        raise FileNotFoundError("DSFB repository not found and AUTO_CLONE_REPO is disabled")

    target = Path(REPO_ROOT).expanduser() if REPO_ROOT is not None else Path("/content/dsfb")
    if target.exists() and not (target / "crates/dsfb-srd/Cargo.toml").exists():
        raise FileExistsError(
            f"Cannot clone DSFB into {target}: path exists but is not the DSFB repository"
        )

    if not target.exists():
        run_checked(["git", "clone", "--depth", "1", DSFB_REPO_URL, str(target)])

    return target

def cargo_env():
    env = os.environ.copy()
    cargo_bin = str(Path.home() / ".cargo" / "bin")
    env["PATH"] = cargo_bin + os.pathsep + env.get("PATH", "")
    return env

def ensure_cargo():
    env = cargo_env()
    if shutil.which("cargo", path=env["PATH"]):
        return env

    run_checked(
        ["sh", "-c", "curl https://sh.rustup.rs -sSf | sh -s -- -y"],
        env=env,
    )

    if not shutil.which("cargo", path=env["PATH"]):
        raise RuntimeError("Rust installation completed but cargo is still unavailable")

    return env

def list_run_dirs(output_root):
    if not output_root.exists():
        return []
    return sorted(path for path in output_root.iterdir() if path.is_dir())

def run_generator_from_scratch(repo_root):
    env = ensure_cargo()
    output_root = repo_root / "output-dsfb-srd"
    before = {path.name for path in list_run_dirs(output_root)}

    run_checked(
        [
            "cargo",
            "run",
            "--manifest-path",
            "crates/dsfb-srd/Cargo.toml",
            "--release",
            "--bin",
            "dsfb-srd-generate",
        ],
        cwd=repo_root,
        env=env,
    )

    after = list_run_dirs(output_root)
    if not after:
        raise FileNotFoundError(
            f"No timestamped runs found under {output_root} after running dsfb-srd-generate"
        )

    fresh_runs = [path for path in after if path.name not in before]
    if not fresh_runs:
        raise RuntimeError(
            "dsfb-srd-generate completed but no new timestamped output directory was created"
        )

    return output_root, fresh_runs[-1]

REPO_ROOT = ensure_repo_root()
OUTPUT_ROOT, RUN_DIR = run_generator_from_scratch(REPO_ROOT)

print(f"Using repository: {REPO_ROOT}")
print(f"Using output root: {OUTPUT_ROOT}")
print(f"Using fresh run directory: {RUN_DIR}")


In [ ]:
manifest = pd.read_csv(RUN_DIR / "run_manifest.csv")
events = pd.read_csv(RUN_DIR / "events.csv")
threshold_sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")
transition_sharpness = pd.read_csv(RUN_DIR / "transition_sharpness.csv")
time_local = pd.read_csv(RUN_DIR / "time_local_metrics.csv")

manifest

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for n_events, subset in threshold_sweep.groupby("n_events"):
    subset = subset.sort_values("tau_threshold")
    ax.plot(
        subset["tau_threshold"],
        subset["reachable_fraction"],
        linewidth=2,
        label=f"N={n_events}",
    )

ax.set_title("Figure 1: Connectivity Collapse $\\rho(\\tau)$")
ax.set_xlabel("Trust threshold $\\tau$")
ax.set_ylabel("Reachable fraction $\\rho(\\tau)$")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for n_events, subset in transition_sharpness.groupby("n_events"):
    subset = subset.sort_values("tau_midpoint")
    ax.plot(
        subset["tau_midpoint"],
        subset["abs_drho_dtau"],
        linewidth=2,
        label=f"N={n_events}",
    )

ax.set_title("Figure 2: Transition Sharpness $|d\\rho / d\\tau|$")
ax.set_xlabel("Threshold midpoint")
ax.set_ylabel("Absolute derivative")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
manifest_row = manifest.iloc[0]
shock_start = manifest_row["shock_start"]
shock_end = manifest_row["shock_end"]

time_local = time_local.copy()
time_local["window_midpoint"] = 0.5 * (time_local["window_start"] + time_local["window_end"])

fig, ax = plt.subplots(figsize=(10, 6))
for tau_threshold, subset in time_local.groupby("tau_threshold"):
    subset = subset.sort_values("window_midpoint")
    ax.plot(
        subset["window_midpoint"],
        subset["reachable_fraction"],
        linewidth=2,
        label=f"tau={tau_threshold:.2f}",
    )

ax.axvspan(shock_start, shock_end, color="tomato", alpha=0.18, label="shock interval")
ax.set_title("Figure 3: Time-Local Connectivity During Structural Regimes")
ax.set_xlabel("Event index")
ax.set_ylabel("Window reachable fraction")
ax.legend()
fig.tight_layout()
plt.show()